# Cyber Week Promotional Post-Mortem

As requested by the VP of Global Sales, this analysis calculates the true Net Profit and Return Rate for the Cyber Week promo (Nov 24 - Nov 30, 2025).

In [1]:
import pandas as pd
import numpy as np
import os

# --- Robust Path Resolution ---
if os.path.exists("/workspace/data"):
    DATA_DIR = "/workspace/data"
    WORKSPACE_DIR = "/workspace"
elif os.path.exists("../environment/data"):
    DATA_DIR = "../environment/data"
    WORKSPACE_DIR = "../workspace"
elif os.path.exists("environment/data"):
    DATA_DIR = "environment/data"
    WORKSPACE_DIR = "workspace"
else:
    DATA_DIR = "data"
    WORKSPACE_DIR = "."

os.makedirs(WORKSPACE_DIR, exist_ok=True)

### 1. Reviewing the Strategic Requirements
Let's review the memo to ensure we capture all business constraints.

In [2]:
memo_path = os.path.join(DATA_DIR, "promo_strategy.txt")
with open(memo_path, 'r') as f:
    print("--- PROMO STRATEGY MEMO ---")
    print(f.read())
    print("---------------------------\n")

--- PROMO STRATEGY MEMO ---
To: Retail Analytics Team
From: VP of Global Sales
Date: December 31, 2025
Subject: Cyber Week Promo Post-Mortem

Team,

As we close out the year, I need a precise financial post-mortem on our "Cyber Week Promo" which ran exclusively from November 24, 2025, through November 30, 2025. 

I need you to calculate the true Net Profit and the Return Rate specifically for orders that originated during this promotional window. 

Please ensure you are looking at our entire global market. North America (NA) is our biggest driver, and I want to make sure their numbers are fully factored into the global net profit. 

Also, remember that Net Profit = (Revenue - COGS) - Refunds. Do not forget to account for items bought during Cyber Week that were returned later in December. 

Thanks,
VP of Global Sales

---------------------------



### 2. Data Ingestion & Cleaning
**CRITICAL EDA NOTE:** The `region` column uses the standard abbreviation "NA" for North America. By default, pandas `read_csv` parses the string "NA" as a missing value (NaN). If we don't explicitly disable this, any subsequent `.dropna()` cleaning will accidentally delete the entire North American market, severely undercounting global profit.

In [3]:
# Load orders carefully to preserve the 'NA' string
orders_df = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'), keep_default_na=False)
returns_df = pd.read_csv(os.path.join(DATA_DIR, 'returns.csv'))

# Ensure correct datatypes
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
returns_df['return_date'] = pd.to_datetime(returns_df['return_date'])

print("Orders Info:")
orders_df.info()
print("\nRegion value counts (verifying North America wasn't dropped):")
print(orders_df['region'].value_counts())

Orders Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    8000 non-null   object        
 1   order_date  8000 non-null   datetime64[ns]
 2   region      8000 non-null   object        
 3   category    8000 non-null   object        
 4   revenue     8000 non-null   float64       
 5   cogs        8000 non-null   float64       
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 375.1+ KB

Region value counts (verifying North America wasn't dropped):
region
NA       3222
EU       2382
APAC     1617
LATAM     779
Name: count, dtype: int64


### 3. Filtering the Promo Window
The promo ran strictly from November 24, 2025, through November 30, 2025.

In [4]:
promo_orders = orders_df[(orders_df['order_date'] >= '2025-11-24') & (orders_df['order_date'] <= '2025-11-30')].copy()
total_promo_orders = int(len(promo_orders))

print(f"Total orders placed during Cyber Week: {total_promo_orders}")

Total orders placed during Cyber Week: 929


### 4. Merging Returns (Accounting for Temporal Lag)
**BUSINESS LOGIC CONSTRAINT:** We cannot simply filter `returns.csv` for November dates. Many returns for Cyber Week purchases occur in December due to shipping and policy windows. We must perform a `left merge` from our base `promo_orders` dataframe to accurately capture all downstream refunds tied to this cohort.

In [5]:
# Left merge to find any return associated with these specific promo orders
promo_performance = promo_orders.merge(returns_df, on='order_id', how='left')

# Fill missing refunds with 0 (items that were not returned)
promo_performance['refund_amount'] = promo_performance['refund_amount'].fillna(0)

print(promo_performance.head())

    order_id order_date region     category  revenue    cogs  return_id  \
0  ORD-10016 2025-11-28     NA         Toys   166.04   81.24  RET-50954   
1  ORD-10031 2025-11-27     NA  Electronics   101.85   56.72        NaN   
2  ORD-10033 2025-11-29     NA         Home   475.47  163.42        NaN   
3  ORD-10043 2025-11-28  LATAM  Electronics   456.47  265.68        NaN   
4  ORD-10052 2025-11-25     EU         Toys   470.96  322.19        NaN   

  return_date  refund_amount  
0  2025-12-09         166.04  
1         NaT           0.00  
2         NaT           0.00  
3         NaT           0.00  
4         NaT           0.00  


### 5. Calculating Final Metrics
Net Profit = (Revenue - COGS) - Refunds.

In [6]:
total_revenue = promo_performance['revenue'].sum()
total_cogs = promo_performance['cogs'].sum()
total_refunds = promo_performance['refund_amount'].sum()

# Final Variables for extraction
global_promo_net_profit = round(float((total_revenue - total_cogs) - total_refunds), 2)

num_returns = promo_performance['return_id'].notna().sum()
promo_return_rate = round(float(num_returns / total_promo_orders), 4)

print(f"Global Promo Net Profit: ${global_promo_net_profit:,.2f}")
print(f"Promo Return Rate: {promo_return_rate * 100:.2f}%")

Global Promo Net Profit: $90,433.04
Promo Return Rate: 12.59%


### 6. Exporting Deliverables
Saving the final metrics to a structured CSV file.

In [7]:
summary_df = pd.DataFrame([{
    'total_promo_orders': total_promo_orders,
    'global_promo_net_profit': global_promo_net_profit,
    'promo_return_rate': promo_return_rate
}])

output_path = os.path.join(WORKSPACE_DIR, 'promo_summary.csv')
summary_df.to_csv(output_path, index=False)
print(f"Success! Summary saved to {output_path}")

Success! Summary saved to ../workspace\promo_summary.csv
